# Dataset Export

**Part 6 of the AQ Integration module.**

Part 5 produced a single, integrated **master dataset**. This notebook
packages that dataset in three interoperable formats so that a partner
may consume the dataset without re-executing the pipeline:

- **CSV** — human readable and supported by ubiquitous tooling.
- **Parquet** — typed, compressed, and suitable for efficient column-pruned reads.
- **Excel** — a spreadsheet-friendly review copy.

The export is performed for both shapes of the master:

- *long* — one row per (time, location_id, variable).
- *wide* — one row per (time, location_id), with one column per variable.

---

## Setup Procedure

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

print(f"pandas {pd.__version__}")

pandas 3.0.3


## Load the Master Dataset (or Synthesise a Stand-in)

If Part 5 produced `master_long.csv` and `master_wide.csv`, those
files are loaded. Otherwise, the notebook synthesises a small
stand-in so that the export steps can be executed.

In [2]:
LONG_PATH = Path("../../data/outputs/master_long.csv")
WIDE_PATH = Path("../../data/outputs/master_wide.csv")

if LONG_PATH.exists() and WIDE_PATH.exists():
    long = pd.read_csv(LONG_PATH, parse_dates=["time"])
    wide = pd.read_csv(WIDE_PATH, parse_dates=["time"])
    print(f"Loaded Part 5 master: long={long.shape}, wide={wide.shape}")
else:
    times = pd.date_range("2024-06-01", "2024-06-30 23:00", freq="1h", tz="UTC")
    stations = pd.read_csv("../../data/stations_example.csv")[["location_id", "lat", "lon"]]
    grid = stations.merge(pd.DataFrame({"time": times}), how="cross")
    rng = np.random.default_rng(0)
    long_rows = []
    for var, lo, hi, src, method in [
        ("pm2p5", 5, 35, "synth_aq", "mean_1h_left"),
        ("t2m",   285, 305, "synth_era5", "bilinear"),
        ("traffic_count_h", 5, 800, "synth_traffic", "colocated"),
    ]:
        v = rng.uniform(lo, hi, size=len(grid))
        long_rows.append(grid.assign(
            variable=var, value=v, source=src, method=method,
            dataset_version="synth",
        ))
    long = pd.concat(long_rows, ignore_index=True)[
        ["time","location_id","lat","lon","variable","value","source","method","dataset_version"]
    ]
    wide = long.pivot_table(
        index=["time","location_id","lat","lon"],
        columns="variable", values="value", aggfunc="first",
    ).reset_index()
    wide.columns.name = None
    print(f"Synthesised stand-in master: long={long.shape}, wide={wide.shape}")
long.head()

Loaded Part 5 master: long=(61215, 9), wide=(3610, 24)


,time,location_id,lat,lon,variable,value,source,method,dataset_version
0,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,pressure_station,1009.275549,station_meteo,colocated,synth-2024-06
1,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,rh_station,69.910070,station_meteo,colocated,synth-2024-06
2,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,temp_station,16.247247,station_meteo,colocated,synth-2024-06
3,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,traffic_count_h,36.000000,traffic_counters,colocated,synth-2024-06
4,2024-05-31 22:00:00+00:00,OSIJEK01,45.5511,18.6939,traffic_heavy_frac,0.129600,traffic_counters,colocated,synth-2024-06


## Export to CSV

In [3]:
OUT_DIR = Path("../../data/outputs/integrated_dataset_v0.1.0")
OUT_DIR.mkdir(parents=True, exist_ok=True)

long_csv = OUT_DIR / "master_long.csv"
wide_csv = OUT_DIR / "master_wide.csv"

long.to_csv(long_csv, index=False, encoding="utf-8")
wide.to_csv(wide_csv, index=False, encoding="utf-8")

print(f"Wrote {long_csv}  ({len(long):,} rows)")
print(f"Wrote {wide_csv}  ({len(wide):,} rows)")

Wrote ..\..\data\outputs\integrated_dataset_v0.1.0\master_long.csv  (61,215 rows)
Wrote ..\..\data\outputs\integrated_dataset_v0.1.0\master_wide.csv  (3,610 rows)


## Export to Parquet

Parquet provides typed, compressed storage and supports column-pruned
reads. 

In [4]:
long_pq = OUT_DIR / "master_long.parquet"
wide_pq = OUT_DIR / "master_wide.parquet"

try:
    import pyarrow  # noqa: F401
    long.to_parquet(long_pq, engine="pyarrow", compression="snappy", index=False)
    wide.to_parquet(wide_pq, engine="pyarrow", compression="snappy", index=False)
    print(f"Wrote {long_pq}  ({len(long):,} rows)")
    print(f"Wrote {wide_pq}  ({len(wide):,} rows)")
except ImportError:
    print("pyarrow not installed - Parquet export skipped. `pip install pyarrow` to enable.")

Wrote ..\..\data\outputs\integrated_dataset_v0.1.0\master_long.parquet  (61,215 rows)
Wrote ..\..\data\outputs\integrated_dataset_v0.1.0\master_wide.parquet  (3,610 rows)


## 5. Export to Excel

For the wide table, a single sheet is written. For the long table, a
single sheet is also written; however, note that Excel caps a sheet
at approximately 1,048,576 rows. If the master dataset grows beyond
that limit, you should rely on the CSV or Parquet exports for the
long shape.

In [5]:
long_xlsx = OUT_DIR / "master_long.xlsx"
wide_xlsx = OUT_DIR / "master_wide.xlsx"

EXCEL_ROW_LIMIT = 1_048_576


def _strip_tz(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_datetime64_any_dtype(s) and getattr(s.dt, "tz", None) is not None:
            out[col] = s.dt.tz_localize(None)
    return out


try:
    import openpyxl  # noqa: F401
    if len(long) < EXCEL_ROW_LIMIT:
        _strip_tz(long).to_excel(long_xlsx, index=False, sheet_name="master_long")
        print(f"Wrote {long_xlsx}  ({len(long):,} rows)")
    else:
        print(f"Skipped {long_xlsx}: {len(long):,} rows exceeds Excel sheet limit.")
    _strip_tz(wide).to_excel(wide_xlsx, index=False, sheet_name="master_wide")
    print(f"Wrote {wide_xlsx}  ({len(wide):,} rows)")
except ImportError:
    print("openpyxl not installed - Excel export skipped. `pip install openpyxl` to enable.")

Wrote ..\..\data\outputs\integrated_dataset_v0.1.0\master_long.xlsx  (61,215 rows)
Wrote ..\..\data\outputs\integrated_dataset_v0.1.0\master_wide.xlsx  (3,610 rows)
